# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library in a reproducible, schema-centric manner referencing all entities by their `@id`.

### Dataset Source
The dataset Croissant schema is available at:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Define the URL for the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', None)}\n\n{getattr(metadata, 'description', None)}\n")

## 2. Data Overview
Review available record sets and fields, referencing all entities by their `@id` fields.

In [ ]:
# List all available record sets and their field @ids
print('Available record sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        # Single field case
        fields = [fields]
    for f in fields:
        f_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
        print(f"    - Field @id: {f_id}")

# Show a preview of the records using record set @id(s)
print('\nSample records from each record set:')
for rs in record_sets:
    rs_id = rs['@id']
    try:
        recs = dataset.records(record_set=rs_id)
        print(f"\nRecord Set @id: {rs_id}")
        for i, rec in enumerate(recs):
            if i >= 2:
                break
            print(rec)
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

## 3. Data Extraction
Load data for each record set into DataFrames. All access is done using record set and field `@id`s.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
print('Extracting records for all record sets...')
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded Record Set: {rs_id} ({len(dataframes[rs_id])} rows)")
        else:
            print(f"No records found for Record Set: {rs_id}")
    except Exception as e:
        print(f"Failed to extract data for {rs_id}: {e}")

# Try to provide a sample columns listing from a main record set, if available
sample_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        sample_record_set_id = k
        break
if sample_record_set_id:
    print(f'\nColumns in {sample_record_set_id}:')
    print(dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head(3))
else:
    print('No populated record sets to sample.')

## 4. Exploratory Data Analysis (EDA)
Apply several processing steps: filter/transform fields using their `@id` and example criteria. All columns, groupings, and filters are referenced by `@id`.

In [ ]:
# If no data is loaded, skip EDA
if not dataframes or sample_record_set_id is None:
    print("No data were loaded for EDA.")
else:
    df = dataframes[sample_record_set_id]
    print(f"Sample record set: {sample_record_set_id}")

    # Identify numeric fields by dtype and @id
    numeric_fields = []
    for col in df.columns:
        # Try to identify numeric fields (float/int) by sample data
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_fields.append(col)
    if not numeric_fields:
        print('No numeric fields found in this record set.')
    else:
        print('Numeric fields found:')
        for nf in numeric_fields:
            print(f"  Field @id: {nf}")

        # Pick the first numeric field for demonstration
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0

        # Example: filter where values > threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nRecords where {numeric_field_id} > {threshold:.2f} (using @id as column):\n")
        display(filtered_df[[numeric_field_id]].head())

        # Normalize the numeric column
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt a groupby on first available non-numeric field (using its @id)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            try:
                grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"\nGrouped mean {numeric_field_id} by {group_field_id} (@id):")
                display(grouped.head())
            except Exception as e:
                print(f"Could not group by {group_field_id}: {e}")
        else:
            print('No suitable non-numeric field available for grouping.')

## 5. Visualization
Visualize distributions or relationships for fields using their `@id`.

In [ ]:
# Plot a histogram for the numeric field (if available)
if sample_record_set_id and numeric_fields:
    fig, ax = plt.subplots(figsize=(7, 4))
    df[numeric_field_id].dropna().hist(ax=ax, bins=20)
    ax.set_title(f"Histogram of field {numeric_field_id} (@id)")
    ax.set_xlabel(numeric_field_id)
    ax.set_ylabel("Frequency")
    plt.show()

    # If group field exists, plot boxplot grouped by it
    if group_field_id and group_field_id in df.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        df.boxplot(column=numeric_field_id, by=group_field_id, ax=ax)
        ax.set_title(f"Boxplot of {numeric_field_id} by {group_field_id} (both @id)")
        ax.set_ylabel(numeric_field_id)
        plt.suptitle("")
        plt.show()
else:
    print('No numeric fields available for plotting.')

## 6. Conclusion
This notebook provided a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, always referencing entities by their `@id` as specified in the Croissant schema. Further analysis can build on these steps for more advanced modeling or insights.
